# Getting started

This notebook is for someone who knows LCA, has used Brightway a little and writes a little Python. It shows how to load data, the main functions, what each one takes and gives back, and the types you will meet. Everything runs on a toy background of three processes, so it needs no licence.

Read it from top to bottom once. Then keep it open as a reference.

## What headwater is

1. A Python library for LCA. It runs inside your Python process: no server and no database to install.
2. A background database such as ecoinvent is **imported once** per version into a **release**: a folder of Parquet files that is never edited, named by a hash of its content.
3. Loading a release solves it once and keeps the score of every product, for every indicator of the method. Your calculations then reuse those scores.
4. Your own model is a **database**: a folder of small YAML and CSV files that lives in git, so every change has an author and a reason. Each of its processes is a product, scored per unit of what it makes. A **study** is a folder next to the databases that asks them a question: which products, how much of each, with which methods and scenarios.
5. Every result can say what it buys, where it is emitted and which lines move it most, and it records what it was computed from.

## Words: LCA, Brightway and headwater

The same idea often has a different name in each tool. The examples come from the toy used below.

| LCA term | Brightway | headwater | Example |
|---|---|---|---|
| Background database | a database in a project: `bd.Database("ecoinvent-3.12-cutoff")` | a **release**: a frozen folder named `<name>@<hash>` | `toy-background@6864b7dbb65d` |
| Where the data lives | a project in your Brightway folder (`BRIGHTWAY2_DIR`) | releases in a cache folder (`HEADWATER_HOME`), databases in git | `~/.headwater/releases/` |
| Activity | an activity (a node), found by its key `(database, code)` | a **process**, found by its id | `toy:fabric-production` |
| Exchange | an exchange (an edge), with a type: production, technosphere, biosphere | a **line**, with a **role**: reference, input, emission, waste_to_treatment and a few more | line `e02`: input, 0.25 kg of cotton fabric |
| Foreground system | a database you write yourself | a **database**: a folder in git | `tshirt` |
| Goal and scope, functional unit | the demand you give `bc.LCA`, and your own notes | a **study**: a folder with `study.yaml`, its products with amounts, methods and scenarios | `tshirt-footprint`: 1 × `tshirt:tshirt` |
| Impact method and category | one method tuple per category: `("EF v3.1", "climate change", "global warming potential (GWP100)")` | one **method** with several **indicators** | method `toy-climate`, indicator `climate-change` |
| Characterisation factor | a `(flow, factor)` pair in a method | a row of the method's `factors.csv` | methane: 29.8 kg CO₂-eq per kg |
| LCA calculation | `bc.LCA({activity: 1}, method)`, `lci()`, `lcia()`, `lca.score` | `db.calc(bg, "tshirt:tshirt")` gives a `Result`, and `result.score` | 1.19205 kg CO₂-eq |
| Many products at once | `MultiLCA` | `hw.score_many` | a table of SKUs |
| Contribution analysis | bw2analyzer, or `lca.characterized_inventory` | `result.where()`: which processes emit; `result.contributions()`: what it buys | electricity emits 55.6% |
| Parameters | project, database and activity parameters | `parameters.yaml` in the database | `fabric_kg = 0.25` |
| Scenario | edit the data, or copy the project | a file in a study's `scenarios/` folder, or an override in the database | `clean-grid` |
| Sensitivity analysis | headwater only | `result.sensitivity()`: each line ranked by how much it moves the score | +1% fabric gives +0.78% |
| Cut-off record | headwater only | a row of `cut_offs.csv`: a line left out on purpose, with its share, reason, who and when | 0.01 kg of packaging tape |
| Flow with no factor | headwater only | a row of `no_factor.csv`: a flow of the database the method does not count, acknowledged | COD to water |
| Results diff | headwater only | `headwater diff`: scores before and after a change, and the lines that caused it | fabric 0.25 → 0.22 kg: -0.111258 |

The [glossary](../docs/appendix/glossary.md) has every term in plain words.

## Set up

This cell finds the repository's folders, makes a temporary work folder, and points `HEADWATER_HOME` at it. `HEADWATER_HOME` is where headwater keeps its releases, `~/.headwater` by default. Using a temporary one here keeps your own cache untouched.

It also defines `headwater(...)`, a small helper that runs the `headwater` command as you would in a terminal and prints what it prints. It shows the work folder as `<work>` and the repository as `<repo>`, hides the time an import takes, and prints the exit code when it is not 0. So the output is the same on every run and on every machine.

In [1]:
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import headwater as hw

here = Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "tests/fixtures/toy").is_dir()), None)
assert repo, "run this notebook from a checkout of the headwater repository"
toy = repo / "tests/fixtures/toy"
demo = repo / "src/headwater/demo"

work = Path(tempfile.mkdtemp(prefix="headwater-")).resolve()
os.environ["HEADWATER_HOME"] = str(work / "home")

HEADWATER = shutil.which("headwater", path=str(Path(sys.executable).parent))


def headwater(*args, cwd=work):
    """Run the headwater command and print what it prints."""
    done = subprocess.run([HEADWATER, *map(str, args)], capture_output=True, text=True, cwd=cwd)
    text = (done.stdout + done.stderr).replace(str(work), "<work>").replace(str(repo), "<repo>")
    print(re.sub(r" in \d+(\.\d+)? s", " in <time>", text).rstrip())
    if done.returncode:
        print(f"(exit code {done.returncode})")

## Loading data

There are four ways to get a background, and one more choice to make: the method.

| Way | What you type | What you get | Example |
|---|---|---|---|
| A folder of processes | `hw.load(folder, method=method_folder, namespace="toy")` | a `Background` | the toy in `tests/fixtures/toy/background` |
| A release folder | `hw.load(release_folder)` | a `Background` | `src/headwater/demo/releases/toy-background@6864b7dbb65d` |
| A release in the cache | `headwater release pull <folder>` once, then `hw.load("name@hash")` | a `Background` | `hw.load("toy-background@6864b7dbb65d")` |
| A source, imported once | `headwater release import <path> --version <version>` | a new release in the cache | ecoinvent's ecoSpold2 folder |

A **`Background`** is a background database, checked and solved once, with a method. It holds the score of every product for every indicator. You make it once per notebook and pass it to everything else.

### A folder of processes

The toy background is a folder of three small CSV files: `processes.csv`, one row per process, `exchanges.csv`, one row per line of a process, and `flows.csv`. A database has the same two tables, with a `database.yaml` beside them. `method` is a method folder. `namespace` is the start of every process id: with `"toy"`, the fabric process is `toy:fabric-production`.

In [2]:
bg = hw.load(toy / "background", method=toy / "methods/toy-climate", namespace="toy")
bg

Background(folder background, namespace toy, method toy-climate 1, 1 indicator, 3 processes)

`hw.load` gave back a `Background`. Printed, it says what it was read from, its method, how many indicators it scores and how many processes it holds. Its attributes give the same in parts: one method, `toy-climate`, with one indicator, `climate-change`.

In [3]:
print("type:      ", type(bg).__name__)
print("method:    ", bg.method.id, bg.method.version)
print("indicators:", [(i.id, i.unit) for i in bg.method.indicators])
print("release:   ", bg.release)

type:       Background
method:     toy-climate 1
indicators: [('climate-change', 'kg CO2-eq')]
release:    None


`release` is `None`: this background was read from a folder, not from a release. Folders are handy for small examples and tests. For real work you use releases.

### A release folder

A **release** is the frozen form of a background. It is a folder named `<name>@<first 12 characters of its hash>`, with Parquet files and a `release.json` that lists the hash of every file. headwater checks those hashes when it opens the folder, so a changed file is an error, not a silent change of your results.

The demo release holds the same toy background, and its method. A release that holds one method needs no `method=`.

In [4]:
release_folder = demo / "releases/toy-background@6864b7dbb65d"
print(sorted(path.name for path in release_folder.iterdir()))

bg = hw.load(release_folder)
print(bg.release)

['exchanges.parquet', 'flow_properties.parquet', 'flows.parquet', 'geographies.parquet', 'issues.parquet', 'methods', 'processes.parquet', 'release.json', 'units.parquet']
Release(toy-background@6864b7dbb65d, source)


This time `bg.release` names the release, with its hash. Every result calculated on this background records that name and hash.

### A release in the cache

You do not want to type a folder path in every notebook. `headwater release pull` checks a release folder and copies it into the cache, `$HEADWATER_HOME/releases/`. Then you load it by its name: `name@hash`, with at least 12 characters of the hash. A team shares releases this way: one person imports, everyone pulls.

In [5]:
headwater("release", "pull", release_folder)

Pulled release toy-background@6864b7dbb65d into <work>/home/releases


In [6]:
bg = hw.load("toy-background@6864b7dbb65d")
print(bg.release)

Release(toy-background@6864b7dbb65d, source)


### A source, imported once

`headwater release import` reads a source database once, links every line to its supplier, checks units and roles, and writes a new release into the cache. You do it once per source and version. After that, nothing reads the source files again.

Today it reads ecoinvent's ecoSpold2 folders, ecoSpold1 files and openLCA JSON-LD packages, such as BAFU's. It detects the format, or you name it with `--format`. SimaPro CSV and ILCD are on the way. For ecoSpold2, `--version` is required, because the files do not record the database version, and the version goes into every process id.

The repository has the toy written as three ecoSpold2 files. Import them like a real ecoinvent folder:

In [7]:
headwater("release", "import", repo / "tests/fixtures/formats/ecospold2/toy", "--version", "1.0")

Read 3 processes of ecoinvent 1.0 cutoff from <repo>/tests/fixtures/formats/ecospold2/toy in <time>: 5 flows, 9 lines.
Lines by role: reference 3, input 1, emission 5.
Wrote release ecoinvent-1.0-cutoff@2605f149739b to <work>/home/releases


The release is called `ecoinvent-1.0-cutoff`, after the source, version and system model. Its process ids have the shape of real ecoinvent ids: `ecoinvent:<version>:cutoff:<activity uuid>_<product uuid>`. `headwater find` searches a release without loading it, so here is the fabric process:

In [8]:
headwater("find", "fabric", "--release", "ecoinvent-1.0-cutoff@2605f149739b")

ecoinvent:1.0:cutoff:a0000000-0000-4000-8000-000000000002_b0000000-0000-4000-8000-000000000001  cotton fabric production | cotton fabric | GLO  kg  (every word in name)


This release has no method yet: ecoSpold2 folders hold inventories, not impact methods. For ecoinvent, `headwater method import` reads ecoinvent's LCIA workbook and writes a method folder. [Notebook 03](03-working-with-ecoinvent.ipynb) does that on the real data.

### The method

`method=` takes one of three things:

- the id of a method the release holds: `method="toy-climate"`
- a method folder: `method=Path("methods/EF-3.1")`
- a list of them, used together: `method=["methods/EF-3.1", "methods/IPCC-2021"]`. The background then scores the indicators of all of them, with one solve.

A method is a folder with a `method.yaml` (its id, version and indicators) and a `factors.csv` (one factor per indicator and flow):

In [9]:
print((toy / "methods/toy-climate/method.yaml").read_text())
print((toy / "methods/toy-climate/factors.csv").read_text())

id: toy-climate
version: "1"
flow_list: toy
indicators:
  - id: climate-change
    name: Climate change
    unit: kg CO2-eq

indicator,flow,location,factor
climate-change,toy:co2,,1
climate-change,toy:ch4,,29.8



## The main functions

You will use these all the time. `bg` is a `Background`, `db` a `Database`, `study` a `Study` and `result` a `Result`.

| Function | What it answers | Inputs (types) | Returns | Example |
|---|---|---|---|---|
| `hw.load(path, method=None, namespace=None)` | Read a background and solve it once | `path`: a release folder, `"name@hash"` or a folder of processes; `method`: an id, a folder, or a list | `Background` | `hw.load("toy-background@6864b7dbb65d")` |
| `bg.find(text, location=None, product=None, kind=None, limit=20)` | Which process is this? | `text`: `str` | tuple of `Match` | `bg.find("fabric")` |
| `bg.score(process_id, indicator=None)` | What is one unit of a background product worth? | `process_id`: `str` | `float` | `bg.score("toy:fabric-production")` |
| `bg.contributions(process_id, indicator=None, min_share=0.03, max_depth=3)` | What is one unit of a background product made of? | `process_id`: `str` | `Contribution` | `bg.contributions("toy:fabric-production")` |
| `bg.where(process_id, indicator=None, top=10)` | Which processes emit its score, and where? | `process_id`: `str` | `Where` | `bg.where("toy:fabric-production")` |
| `bg.sensitivity(process_id, indicator=None, top=None, include_reference=False)` | Which lines move its score most? | `process_id`: `str` | tuple of `Sensitivity` | `bg.sensitivity("toy:fabric-production", top=3)` |
| `hw.open(path)` | Read a database folder, or a study folder | `path`: `str` or `Path` | `Database` or `Study` | `hw.open("tshirt")` |
| `db.validate(bg)` | Is the database correct, without calculating? | `bg`: `Background` | `Report` | `db.validate(bg).errors` |
| `db.calc(bg, process_id, indicator=None)` | The score of one unit of what a process of the database makes | `bg`; `process_id`, `indicator`: `str` | `Result` | `db.calc(bg, "tshirt:tshirt").score` |
| `db.scores(bg, process_id, indicators=None)` | Every indicator at once, just the numbers | `bg`; `process_id`: `str` | `dict` of indicator → `float` | `db.scores(bg, "tshirt:tshirt")` |
| `hw.Study(products, methods, databases=None, scenarios=None)` | A study in memory: products with amounts, asked of the databases in a folder | `products`: `dict` of process id → amount; `methods`: `list` of ids; `databases`: a folder | `Study` | `hw.Study({"tshirt:tshirt": 1}, ["toy-climate"], databases="tshirt-dyed")` |
| `study.calc(bg, indicator=None, scenario=None)` | The score of each product of the study, for its amount, in a scenario or not | `bg`; `indicator`, `scenario`: `str` | tuple of `Result`, one per product | `study.calc(bg, scenario="more-fabric")[0]` |
| `study.save(path, goal=..., functional_unit=..., boundary=..., cut_off=None)` | Write a study made in memory to a folder next to its databases | `path`; the goal and scope: `str` | `Study` | [notebook 02](02-writing-a-database.ipynb) |
| `result.contributions(min_share=0.03, max_depth=3)` | What does the product buy, and what is each part worth? | numbers | `Contribution` | `print(result.contributions())` |
| `result.where(top=10)` | Which processes emit the score, anywhere upstream, and where? | `int` or `None` | `Where` | `print(result.where())` |
| `result.sensitivity(top=None, include_reference=False)` | Which lines move the score most? | `int` or `None` | tuple of `Sensitivity` | `result.sensitivity(top=5)` |
| `result.parameters(top=None)` | Which named parameters move the score most? | `int` or `None` | tuple of `ParameterSensitivity` | `result.parameters()` |
| `hw.score_many(source, table, indicators=None)` | The scores of many products at once | `source`: a `Background` or an indicators release; `table`: columns `sku`, `product_id`, `amount` | `ManyScores` | below |
| `headwater diff REF [REF2] DATABASE` | What changed between two versions of a database, and why? | git refs and a database folder | text, or JSON with `--json` | `headwater diff main tshirt` |

`indicator` can be left out when the method has one indicator. With several, you name one: `indicator="climate-change"`.

The cells below call each one on the toy and show what comes back.

### `bg.find`: find a process

`find` matches the text against process names, then reference product names, by exact rules: the name is the text, starts with it, or has every word of it. Nothing is fuzzy. Each `Match` says which rule matched in `why`. The `process_id` is what you write in a database file.

In [10]:
matches = bg.find("fabric")
matches

(Match(toy:fabric-production: cotton fabric production | cotton fabric | GLO, kg, every word in name),)

In [11]:
for match in bg.find("electricity", location="GLO"):
    print(match.process_id, "|", match.name, "|", match.reference_product, "|", match.location)
    print("  unit:", match.unit, " kind:", match.kind, " why:", match.why)

toy:electricity-production | electricity production | electricity | GLO
  unit: kWh  kind: ordinary_transforming_activity  why: name starts with


### `bg.score`: the score of one unit of a background product

`score` gives the stored score of one unit of what a process makes, as a number: 1 kg of cotton fabric is 3.7086 kg CO₂-eq. `hw.load` already solved every product, so it solves nothing. A waste treatment scores per kg treated, a positive burden: 1.2 for incineration.

In [12]:
bg.score("toy:fabric-production")

3.7086000000000006

### `bg.contributions`: what one unit of a background product is made of

`contributions` returns a `Contribution`: a tree. Each node is a process with its score, its direct emissions, the purchases worth showing (`children`) and a `rest` for the small ones. Printing it shows the tree. Shares are of the top line.

In [13]:
tree = bg.contributions("toy:fabric-production")
print(tree)

1 kg of cotton fabric production: 3.70860 (100.0%), direct 2.11920
  3 kWh of electricity production: 1.58940 (42.9%), direct 1.58940


1 kg of fabric is 3.7086 kg CO₂-eq: 2.1192 of its own emissions, plus 3 kWh of electricity at 0.5298 per kWh. Each node is an object with attributes, so you can use the numbers in code:

In [14]:
print(f"{tree.process_id}: {tree.amount:g} {tree.unit}")
print(f"  score {tree.score:.5f}, of which direct {tree.direct:.5f}")
for child in tree.children:
    print(
        f"  buys {child.amount:g} {child.unit} of {child.process_id} "
        f"at {child.per_unit:.5f} per {child.unit} = {child.score:.5f}"
    )

toy:fabric-production: 1 kg
  score 3.70860, of which direct 2.11920
  buys 3 kWh of toy:electricity-production at 0.52980 per kWh = 1.58940


### `bg.where`: which processes emit, and where

`contributions` follows what a product **buys**. `where` says which processes **emit**, anywhere upstream, and sums them by location. For fabric they agree, because the chain is short. In ecoinvent they differ a lot. It returns a `Where`, with `processes` (each an `Emitter`) and `locations` (each a `Place`).

In [15]:
print(bg.where("toy:fabric-production"))

1 kg of cotton fabric production: 3.70860
By process:
  1 kg of cotton fabric production (GLO): 2.11920 (57.1%)
  3 kWh of electricity production (GLO): 1.58940 (42.9%)
By location:
  GLO: 3.70860 (100.0%), 2 processes


### `bg.sensitivity`: which lines move the score

A sensitivity analysis ranks every line of every process, anywhere in the supply chain, that someone could change. For each line, `sensitivity` is the change of the score per one more unit of the line, and `elasticity` is the % change of the score per 1% change of the line. They come back largest elasticity first.

In [16]:
for line in bg.sensitivity("toy:fabric-production", top=3):
    print(
        f"{line.process_id} line {line.line_id} ({line.role.value}, "
        f"{line.amount:g} {line.unit} {line.flow_id}): "
        f"elasticity {line.elasticity:.4f}, sensitivity {line.sensitivity:.4f}"
    )

toy:fabric-production line e03 (emission, 2 kg toy:co2): elasticity 0.5393, sensitivity 1.0000
toy:fabric-production line e02 (input, 3 kWh toy:electricity): elasticity 0.4286, sensitivity 0.5298
toy:electricity-production line e02 (emission, 0.5 kg toy:co2): elasticity 0.4045, sensitivity 3.0000


The fabric's own 2.0 kg of CO₂ matters most: 1% more of it gives 0.54% more score.

### `hw.open` and `db.validate`: read and check a database

A database is a folder of your own processes. `hw.open` reads its files and checks them on their own: a missing field or a misspelt file name stops here. It returns a `Database`.

`db.validate(bg)` checks the database against the background without calculating: every supplier exists and makes the flow bought, every unit converts, every flow of the database has a factor. It returns a `Report` with `errors` and `warnings`. An empty report means `calc` will not refuse the database.

The demo T-shirt pins the demo release in its `database.yaml`. Its process is one row of `processes.csv`, and its lines are rows of `exchanges.csv`:

In [17]:
database_folder = demo / "studies/tshirt"
print((database_folder / "database.yaml").read_text())
print((database_folder / "processes.csv").read_text())
print((database_folder / "exchanges.csv").read_text())

name: tshirt
background: toy-background@6864b7dbb65d
methods: [toy-climate]

id,name,location,kind
tshirt,T-shirt,GLO,ordinary_transforming_activity

process,line,role,flow,amount,unit,supplier
tshirt,e01,reference,tshirt:tshirt,1,unit,
tshirt,e02,input,toy:cotton-fabric,0.25,kg,toy:fabric-production
tshirt,e03,input,toy:electricity,0.5,kWh,toy:electricity-production



In [18]:
db = hw.open(database_folder)
print(db)
report = db.validate(bg)
print(db.name, report)

Database(tshirt: 1 process, background toy-background@6864b7dbb65d, methods toy-climate)
tshirt Report(no problems)


### `db.calc`: the score

`calc` scores one unit of what a process of the database makes. You name the process by its id: `tshirt:tshirt` is one T-shirt. It does not solve the background again: each of the T-shirt's purchases is the amount times the stored score of what is bought. It returns a `Result`.

In [19]:
result = db.calc(bg, "tshirt:tshirt")
result

Result(tshirt:tshirt = 1.19205 kg CO2-eq, indicator climate-change, method toy-climate 1)

The useful attributes of a `Result`:

- `score`, `unit`, `indicator`, `process_id`: the number and what it is.
- `provenance`: what it was computed from. Among others: the release with its hash, the method and its version and hash, the `background:` pin of the database, the solver settings, the residual (how well the equations are met), and `cancellation` (below).
- `not_characterised`: flows of the background that the database reaches and the method does not count. They count 0, and are listed so nothing is dropped without a trace. The toy has none.

In [20]:
print("score:     ", result.score, result.unit)
print("indicator: ", result.indicator)
print("process:   ", result.process_id)
p = result.provenance
print("release:   ", [f"{r.name}@{r.hash[:12]}" for r in p.releases])
print("method:    ", p.method, p.method_version)
print("pin:       ", p.pin)
print("residual below tolerance:", p.residual <= p.solver.tolerance)
print("not characterised:", result.not_characterised)
assert abs(result.score - 1.19205) < 1e-9 * 1.19205

score:      1.19205 kg CO2-eq
indicator:  climate-change
process:    tshirt:tshirt
release:    ['toy-background@6864b7dbb65d']
method:     toy-climate 1
pin:        toy-background@6864b7dbb65d
residual below tolerance: True
not characterised: ()


`provenance.cancellation` holds the positive and negative parts of the score. A score near 0 can hide large parts that cancel, for example biogenic carbon taken up and given back. `cancels` flags it. The T-shirt has no negative part:

In [21]:
print(p.cancellation, "cancels:", p.cancellation.cancels)

Cancellation(positive=1.19205, negative=0.0) cancels: False


`db.scores(bg, "tshirt:tshirt")` gives every indicator at once, as a plain dictionary. It holds only the numbers: no contributions and no provenance.

In [22]:
db.scores(bg, "tshirt:tshirt")

{'climate-change': 1.19205}

### `result.contributions`, `result.where`, `result.sensitivity`: why this number

The same three questions as for a background product, now for the T-shirt. None of them solves anything again.

In [23]:
print(result.contributions())

1 unit of T-shirt: 1.19205 (100.0%), direct 0.00000
  0.25 kg of cotton fabric production: 0.92715 (77.8%), direct 0.52980
    0.75 kWh of electricity production: 0.39735 (33.3%), direct 0.39735
  0.5 kWh of electricity production: 0.26490 (22.2%), direct 0.26490


In [24]:
print(result.where())

1 unit of T-shirt: 1.19205
By process:
  1.25 kWh of electricity production (GLO): 0.66225 (55.6%)
  0.25 kg of cotton fabric production (GLO): 0.52980 (44.4%)
By location:
  GLO: 1.19205 (100.0%), 2 processes


`contributions` shows the T-shirt's own electricity as 0.26490: what the T-shirt buys. `where` shows electricity production emitting 0.66225: all 1.25 kWh it makes for the T-shirt, 0.5 bought directly and 0.75 through the fabric.

In [25]:
for line in result.sensitivity(top=3):
    print(
        f"{line.process_id} line {line.line_id}: elasticity {line.elasticity:.4f}, "
        f"sensitivity {line.sensitivity:.4f}"
    )

tshirt:tshirt line e02: elasticity 0.7778, sensitivity 3.7086
toy:electricity-production line e02: elasticity 0.5243, sensitivity 1.2500
toy:fabric-production line e03: elasticity 0.4194, sensitivity 0.2500


### `result.parameters`: which named parameters matter

A database can name its numbers in `parameters.yaml` and use the names in its lines ([notebook 02](02-writing-a-database.ipynb)). `result.parameters()` ranks them as `sensitivity` ranks lines. This toy database writes its fabric as `fabric_kg`. The study next to it, `tshirt-footprint`, has a scenario that sets it to 0.30 kg. A study gives one `Result` per product, so `[0]` is the T-shirt:

In [26]:
parameter_db = hw.open(toy / "tshirt-parameter/tshirt")
toy_bg = hw.load(toy / "background", method=toy / "methods/toy-climate", namespace="toy")
for parameter in parameter_db.calc(toy_bg, "tshirt:tshirt").parameters():
    print(
        f"{parameter.parameter_id} = {parameter.value:g} {parameter.unit}: "
        f"elasticity {parameter.elasticity:.4f}, sensitivity {parameter.sensitivity:.4f}"
    )
study = hw.open(toy / "tshirt-parameter/tshirt-footprint")
more_fabric = study.calc(toy_bg, scenario="more-fabric")[0]
print(more_fabric)
assert abs(more_fabric.score - 1.37748) < 1e-9 * 1.37748

fabric_kg = 0.25 kg: elasticity 0.7778, sensitivity 3.7086
Result(tshirt:tshirt = 1.37748 kg CO2-eq, indicator climate-change, method toy-climate 1, scenario more-fabric)


This database pins `toy-background@fixture`, a label and not a release, so it is calculated on the toy folder. A database that pins a release is refused on any other release: the pin makes sure a number is always calculated on the data it was written for.

### `hw.score_many`: many products at once

A SKU is one product you sell, described by what it buys from the background. `score_many` takes a table with one row per SKU and product, and scores every SKU for every indicator with one multiplication. The table can be a dictionary of columns, as here, or an Arrow table. It returns a `ManyScores`.

In [27]:
skus = hw.score_many(
    bg,
    {
        "sku": ["shirt-a", "shirt-a", "shirt-b"],
        "product_id": [
            "toy:fabric-production",
            "toy:electricity-production",
            "toy:fabric-production",
        ],
        "amount": [0.25, 0.5, 0.30],
    },
)
print(skus.skus, skus.indicators, skus.units)
print(skus["climate-change"])

('shirt-a', 'shirt-b') ('climate-change',) ('kg CO2-eq',)
[1.19205 1.11258]


`shirt-a` is the T-shirt again: 1.19205. `shirt-b` buys 0.30 kg of fabric only: 0.30 × 3.7086 = 1.11258.

### `headwater diff`: what changed, and why

Databases live in git, so a change is a commit. `headwater diff` calculates a database at two git refs and lists the lines that changed, ranked by their effect on the score. CI posts the same report on every pull request that changes a database.

This cell makes a throwaway git repository with the demo T-shirt in it and commits it. The author and the dates are fixed so the commit hash is the same on every run.

In [28]:
databases_repo = work / "databases-repo"
shutil.copytree(database_folder, databases_repo / "tshirt")
identity = {
    "GIT_AUTHOR_NAME": "Notebook",
    "GIT_AUTHOR_EMAIL": "notebook@example.com",
    "GIT_AUTHOR_DATE": "2026-10-01T12:00:00Z",
    "GIT_COMMITTER_NAME": "Notebook",
    "GIT_COMMITTER_EMAIL": "notebook@example.com",
    "GIT_COMMITTER_DATE": "2026-10-01T12:00:00Z",
}
for command in (
    ["init", "-q", "-b", "main"],
    ["add", "."],
    ["-c", "commit.gpgsign=false", "commit", "-q", "-m", "T-shirt"],
):
    subprocess.run(
        ["git", *command], cwd=databases_repo, env={**os.environ, **identity}, check=True
    )

Now change the fabric from 0.25 to 0.22 kg, without committing, and ask what that does. With one ref, `diff` compares that ref with the files as they are now.

In [29]:
csv = databases_repo / "tshirt/exchanges.csv"
csv.write_text(csv.read_text().replace(",0.25,", ",0.22,"))
headwater("diff", "main", "tshirt", cwd=databases_repo)

Database tshirt: main (67b8eef3b381) → working tree
Background: toy-background@6864b7dbb65d, method toy-climate 1

T-shirt (tshirt:tshirt)
  Climate change: 1.19205 → 1.08079 kg CO2-eq, change -0.111258 (-9.3%)

Lines that differ, ranked by their effect on Climate change (kg CO2-eq):
  -0.111258  tshirt:tshirt line e02 (cotton fabric): amount 0.25 → 0.22 kg


The score drops by 0.111258, and line e02 is the cause: 0.03 kg less fabric at 3.7086 per kg.

## The types you will meet

### Ids

Every process and flow has an id: a text that never changes. Ids link everything. Names are for people.

| What | Shape | Example |
|---|---|---|
| A process of an ecoinvent release | `ecoinvent:<version>:cutoff:<activity uuid>_<product uuid>` | `ecoinvent:1.0:cutoff:a0000000-..._b0000000-...` above |
| A process of a folder background | `<namespace>:<id>`, with the `id` of its row in `processes.csv` | `toy:fabric-production` |
| A process of a database | `<database>:<id>`, with the `id` of its row in `processes.csv` | `tshirt:tshirt` |
| The database's own copy of a background process, made by an override | `<database>:copy:<background id>` | `tshirt:copy:toy:electricity-production` |
| A product flow of ecoinvent | `ecoinvent:product:<uuid>` | |
| An elementary flow of ecoinvent | `ecoinvent:flow:<uuid>` | |
| A flow of the toy or of a database | `<namespace>:<name>` | `toy:co2`, `tshirt:tshirt` |
| A line | the process id plus a line id | `tshirt:tshirt line e02` |

The version is part of an ecoinvent process id, so ids from 3.11 and 3.12 never mix by mistake. A product is named by the id of the process that makes it: `bg.contributions("toy:fabric-production")` is 1 kg of cotton fabric.

### Amounts and units

Amounts are Python `float`s (64-bit). Each flow has a reference unit, and every score is per unit of it: per kg of fabric, per kWh of electricity, per kg of waste **treated**. A database line may use another unit of the same kind, such as `g` for a flow in kg; headwater converts it. headwater writes units short (`kg`, `kWh`), where Brightway writes them long (`kilogram`, `kilowatt hour`).

Amounts in files are positive. The role says what the line means, and headwater turns roles into matrix signs in one place. A waste sent to treatment is a positive amount with the role `waste_to_treatment`, and its burden counts up.

### Tables: Arrow, and pandas if you want it

Large results come back as Apache Arrow tables (`pyarrow.Table`). They need no pandas. `.to_pandas()` turns one into a pandas DataFrame, if pandas is installed: the `notebooks` dependency group installs it, headwater itself does not need it.

In [30]:
table = skus.to_arrow()
print(type(table).__name__, table.schema.names)
table.to_pandas()

Table ['sku', 'climate-change']


,sku,climate-change
0,shirt-a,1.19205
1,shirt-b,1.11258


A release's own tables are Arrow too: `bg.release.table("processes")`. `bg.release.connect()` opens them in DuckDB, so you can ask questions in SQL:

In [31]:
con = bg.release.connect()
con.sql("select id, name, location from processes order by id").df()

,id,name,location
0,toy:electricity-production,electricity production,GLO
1,toy:fabric-production,cotton fabric production,GLO
2,toy:incineration-textile-waste,incineration of textile waste,GLO


### Errors

headwater stops with an error rather than return a number it cannot vouch for. Two kinds come up most:

- **`InvalidDataError`**: the files or records are wrong. It carries **every** problem found, not only the first, in `.problems`. Each problem has a `code` that tools can match on, an `item` that names what is wrong, a `message` that says why and often how to fix it, and a `location` (file and row) when known.
- **`ValueError`**: an argument is wrong, such as an id the background does not hold. The message suggests the closest one.

Every error headwater raises because of the data is a `hw.HeadwaterError`, so `except hw.HeadwaterError` catches them all. A wrong argument is Python's own `ValueError`.

Trigger one on purpose: copy the T-shirt and misspell its fabric supplier and unit.

In [32]:
broken = work / "broken/tshirt"
shutil.copytree(database_folder, broken)
lines = broken / "exchanges.csv"
lines.write_text(lines.read_text().replace(",kg,toy:fabric-production", ",kgs,toy:fabric-prod"))

try:
    hw.open(broken).calc(bg, "tshirt:tshirt")
except hw.InvalidDataError as error:
    print(error)
    print()
    for problem in error.problems:
        print("code:   ", problem.code)
        print("item:   ", problem.item)
        print("message:", problem.message)

2 problems:
- tshirt:tshirt line e02: unknown unit 'kgs' (flow toy:cotton-fabric). Did you mean 'kg'?
- tshirt:tshirt line e02: supplier 'toy:fabric-prod' does not exist (flow 'toy:cotton-fabric'). Did you mean 'toy:fabric-production'?

code:    unknown-unit
item:    tshirt:tshirt line e02
message: unknown unit 'kgs' (flow toy:cotton-fabric). Did you mean 'kg'?
code:    supplier-not-found
item:    tshirt:tshirt line e02
message: supplier 'toy:fabric-prod' does not exist (flow 'toy:cotton-fabric'). Did you mean 'toy:fabric-production'?


Both problems came back at once, each with a suggestion. `db.validate(bg)` returns the same problems in a `Report`, without raising, which is handy in a loop over many databases.

And a `ValueError`, from an id that does not exist:

In [33]:
try:
    bg.contributions("toy:fabric")
except ValueError as error:
    print(error)

'toy:fabric' is not the id of a process of this background. Did you mean 'toy:fabric-production'?


## Where to go next

- [01-first-footprint](01-first-footprint.ipynb): a footprint step by step, what each number means, and how to change one.
- [02-writing-a-database](02-writing-a-database.ipynb): every file of a database, one by one, and the mistakes headwater catches.
- [03-working-with-ecoinvent](03-working-with-ecoinvent.ipynb): the same on ecoinvent 3.12. Needs a licence.
- [04-coming-from-brightway](04-coming-from-brightway.ipynb): the same tasks in Brightway and in headwater, side by side.
- [05-brightway-and-headwater-on-ecoinvent](05-brightway-and-headwater-on-ecoinvent.ipynb): that comparison on ecoinvent 3.12, timed. Needs a licence.

The design is in [docs/design.md](../docs/design.md), the calculation in [docs/calculation.md](../docs/calculation.md), and the command line in [docs/workflows.md](../docs/workflows.md).